In [1]:
import numpy as np
import pandas as pd
import sklearn

from tools import classification_tools as tools
from src.simple_sklearn.classification.one_r import OneRClassifier
from src.simple_sklearn.classification.naive_bayes import NaiveBayesClassifier
from src.simple_sklearn.classification.decision_tree import DecisionTreeClassifier
from src.simple_sklearn.classification.k_neighbors import KNeighborsClassifier

### data_4x

In [2]:
df = pd.read_csv('../data/classification_4f.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   x0      10 non-null     int64
 1   x1      10 non-null     int64
 2   x2      10 non-null     int64
 3   x3      10 non-null     int64
 4   y       10 non-null     int64
dtypes: int64(5)
memory usage: 528.0 bytes


In [3]:
X, y = df.drop(columns='y'), df['y']
X.shape, y.shape

((10, 4), (10,))

In [4]:
y.replace(0, 'y0', inplace=True)
y.replace(1, 'y1', inplace=True)
y = list(y)

In [5]:
X_pred = pd.DataFrame({'x0': [2], 'x1': [1], 'x2': [1], 'x3': [1]})

#### OneRClassifier

In [6]:
clf = OneRClassifier()
clf.fit(X, y);

In [7]:
tools.OneRClassifier_info(clf)

x[3] == 0: y = y1
x[3] == 1: y = y0


In [8]:
clf.predict(X_pred)

array(['y0'], dtype='<U2')

#### NaiveBayesClassifier

In [9]:
clf = NaiveBayesClassifier()
clf.fit(X, y);

In [10]:
tools.NaiveBayesClassifier_info(clf)

class_log_prior_:
-0.916290731874155 -0.5108256237659907 

feature_log_prob_:
y  feat_v
0  0        -0.559616
   1        -1.252763
1  2        -0.810930
   0        -1.098612
   1        -1.504077
Name: count, dtype: float64 y  feat_v
0  0        -0.693147
   1        -0.693147
1  0        -0.693147
   1        -0.693147
Name: count, dtype: float64 y  feat_v
0  1        -0.405465
   0        -1.098612
1  0        -0.693147
   1        -0.693147
Name: count, dtype: float64 y  feat_v
0  1        -0.405465
   0        -1.098612
1  0        -0.287682
   1        -1.386294
Name: count, dtype: float64 

feature_unknown_log_probs_:
y
0   -1.945910
1   -2.197225
Name: feat_v, dtype: float64 y
0   -1.791759
1   -2.079442
Name: feat_v, dtype: float64 y
0   -1.791759
1   -2.079442
Name: feat_v, dtype: float64 y
0   -1.791759
1   -2.079442
Name: feat_v, dtype: float64 



In [11]:
y_pred = clf.predict(X_pred)
y_pred

array(['y1'], dtype='<U2')

Compare with similar scikit-learn model

In [12]:
sk_clf = sklearn.naive_bayes.CategoricalNB()
sk_clf.fit(X, y);

In [13]:
tools.sklearn_CategoricalNB_info(sk_clf)

class_log_prior_:
-0.9162907318741553 -0.5108256237659909 

feature_log_prob_:
[[-0.55961579 -1.25276297 -1.94591015]
 [-1.09861229 -1.5040774  -0.81093022]] [[-0.69314718 -0.69314718]
 [-0.69314718 -0.69314718]] [[-1.09861229 -0.40546511]
 [-0.69314718 -0.69314718]] [[-1.09861229 -0.40546511]
 [-0.28768207 -1.38629436]] 



In [14]:
sk_y_pred = sk_clf.predict(X_pred)
sk_y_pred

array(['y1'], dtype='<U2')

In [15]:
assert y_pred == sk_y_pred

#### DecisionTreeClassifier

In [16]:
clf = DecisionTreeClassifier()
clf.fit(X, y);

In [17]:
tools.DecisionTreeClassifier_info(clf)

└── Feature[1]
    (0)
    ├── Feature[4]
    │   (0)
    │   ├── Feature[2]
    │   │   (0)
    │   │   ├── Label: 0
    │   │   (1)
    │   │   └── Label: 1
    │   (1)
    │   └── Label: 0
    (1)
    ├── Feature[2]
    │   (0)
    │   ├── Feature[3]
    │   │   (0)
    │   │   ├── Feature[4]
    │   │   │   (1)
    │   │   │   ├── Label: 1
    │   │   │   (0)
    │   │   │   └── Label: 1
    │   │   (1)
    │   │   └── Label: 1
    │   (1)
    │   └── Label: 1
    (2)
    └── Label: 1


In [18]:
clf.predict(X_pred)

array(['y1'], dtype='<U2')

#### KNeighborsClassifier

In [19]:
n_neighbors = 3

##### weights='uniform'

In [20]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors)
clf.fit(X, y);

In [21]:
neigh_dist, neigh_ind = clf.kneighbors(X_pred)
tools.KNeighborsClassifier_info(clf, X_pred)

neighbors_dist:
[1.         1.41421356 1.73205081] 

neighbors_indices:
[8 2 1] 



In [22]:
y_pred = clf.predict(X_pred)
y_pred

array(['y1'], dtype='<U2')

Compare with similar scikit-learn model

In [23]:
sk_clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
sk_clf.fit(X, y);

In [24]:
sk_neigh_dist, sk_neigh_ind = clf.kneighbors(X_pred)
tools.sklearn_KNeighborsClassifier_info(sk_clf, X_pred)

neighbors_dist:
[1.         1.41421356 1.73205081] 

neighbors_indices:
[8 2 1] 



In [25]:
sk_y_pred = sk_clf.predict(X_pred)

In [26]:
assert y_pred == sk_y_pred
assert np.allclose(neigh_dist, sk_neigh_dist)
assert np.all(neigh_ind == sk_neigh_ind)

##### weights='distance'

In [27]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
clf.fit(X, y);

In [28]:
clf.predict(X_pred)

array(['y1'], dtype='<U2')

##### weights='distance_squared'

In [29]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance_squared')
clf.fit(X, y);

In [30]:
clf.predict(X_pred)

array(['y1'], dtype='<U2')

### More datasets (data_3x)

In [31]:
classifiers = [
    OneRClassifier(),
    NaiveBayesClassifier(),
    DecisionTreeClassifier(),
    KNeighborsClassifier(weights='uniform'),
    KNeighborsClassifier(weights='distance'),
    KNeighborsClassifier(weights='distance_squared'),
]
sk_classifiers = [
    sklearn.naive_bayes.CategoricalNB(),
    sklearn.tree.DecisionTreeClassifier(criterion='entropy', random_state=42),
    sklearn.neighbors.KNeighborsClassifier(weights='uniform'),
    sklearn.neighbors.KNeighborsClassifier(weights='distance'),

    sklearn.dummy.DummyClassifier(),
]

In [32]:
df = pd.read_csv('../data/classification_3f.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   dataset  40 non-null     int64
 1   x0       40 non-null     int64
 2   x1       40 non-null     int64
 3   x2       40 non-null     int64
 4   y        40 non-null     int64
dtypes: int64(5)
memory usage: 1.7 KB


In [33]:
res_df = pd.DataFrame()
X_pred = pd.DataFrame({'x0': [1], 'x1': [1], 'x2': [1]})
for clf in classifiers:
    for df_i, df_v in df.groupby('dataset'):
        df_v = df_v.drop(columns=['dataset'])
        X, y = df_v.drop(columns='y'), df_v['y']
        clf.fit(X, y)
        y_pred = clf.predict(X_pred)[0]
        res_df.loc[str(clf), df_i] = y_pred

res_df.astype(int)

,1,2,3,4
OneRClassifier(),1,1,1,1
NaiveBayesClassifier(),0,1,0,1
DecisionTreeClassifier(),0,1,0,1
KNeighborsClassifier(),1,1,0,1
KNeighborsClassifier(weights='distance'),1,1,0,1
KNeighborsClassifier(weights='distance_squared'),1,1,0,1


### Student performance

In [34]:
df = pd.read_csv('../data/student_performance.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2392 entries, 0 to 2391
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   StudentID          2392 non-null   int64  
 1   Age                2392 non-null   int64  
 2   Gender             2392 non-null   int64  
 3   Ethnicity          2392 non-null   int64  
 4   ParentalEducation  2392 non-null   int64  
 5   StudyTimeWeekly    2392 non-null   float64
 6   Absences           2392 non-null   int64  
 7   Tutoring           2392 non-null   int64  
 8   ParentalSupport    2392 non-null   int64  
 9   Extracurricular    2392 non-null   int64  
 10  Sports             2392 non-null   int64  
 11  Music              2392 non-null   int64  
 12  Volunteering       2392 non-null   int64  
 13  GPA                2392 non-null   float64
 14  GradeClass         2392 non-null   float64
dtypes: float64(3), int64(12)
memory usage: 280.4 KB


In [35]:
X = df[[
    'Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'Absences', 'Tutoring',
    'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering'
]]
y = df['GradeClass']
X.shape, y.shape

((2392, 11), (2392,))

In [36]:
from sklearn.model_selection import train_test_split

test_size_ratio = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size_ratio,
    stratify=y,
    random_state=42
)

print(f"Train: X={X_train.shape} y={y_train.shape}")
print(f"Test: X={X_test.shape} y={y_test.shape}")

Train: X=(1913, 11) y=(1913,)
Test: X=(479, 11) y=(479,)


In [37]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)

In [38]:
from sklearn.metrics import accuracy_score

res_df = pd.DataFrame()
for i, clf in enumerate(classifiers + sk_classifiers):
    res_df.loc[i, 'clf'] = str(clf)
    res_df.loc[i, 'impl'] = 'custom' if i < len(classifiers) else 'sklearn'
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    res_df.loc[i, 'accuracy'] = accuracy
res_df.sort_values(by=['accuracy'], ascending=False, inplace=True)

res_df

,clf,impl,accuracy
1,NaiveBayesClassifier(),custom,0.680585
6,CategoricalNB(),sklearn,0.680585
3,KNeighborsClassifier(),custom,0.665971
4,KNeighborsClassifier(weights='distance'),custom,0.661795
8,KNeighborsClassifier(),sklearn,0.661795
5,KNeighborsClassifier(weights='distance_squared'),custom,0.659708
9,KNeighborsClassifier(weights='distance'),sklearn,0.659708
0,OneRClassifier(),custom,0.640919
7,"DecisionTreeClassifier(criterion='entropy', ra...",sklearn,0.582463
2,DecisionTreeClassifier(),custom,0.576200


### Check classifiers (scikit-learn)

In [39]:
classifiers = [
    OneRClassifier(),
    NaiveBayesClassifier(),
    DecisionTreeClassifier(),
    KNeighborsClassifier(),
]

In [40]:
from sklearn.utils.estimator_checks import estimator_checks_generator

for classifier in classifiers:
    total_checks = 0
    for (estimator, check) in estimator_checks_generator(classifier):
        total_checks += 1
        check(estimator)
    print(f"{classifier}: {total_checks} checks complete.")

OneRClassifier(): 55 checks complete.
NaiveBayesClassifier(): 55 checks complete.
DecisionTreeClassifier(): 55 checks complete.
KNeighborsClassifier(): 55 checks complete.
